# 🌊 Water Quality Data Ingestion

## Purpose
Loads water quality data from CSV into Unity Catalog table with data quality checks.

## Steps
1. Load water quality dataset from CSV
2. Perform data quality validation
3. Engineer basic features
4. Create Unity Catalog table with primary key constraint
5. Enable Change Data Feed for monitoring

In [ ]:
# 📦 Install required packages
%pip install pandas numpy databricks-sdk

# Restart Python after install
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries and Setup
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, monotonically_increasing_id
from pyspark.sql.types import *

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Get configuration parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")
schema_name = spark.conf.get("schema_name", "default")

print("✅ Libraries loaded successfully!")
print(f"🎯 Target Catalog: {catalog_name}.{schema_name}")

In [ ]:
# 📊 Load Water Quality Dataset
DATA_PATH = f"/Volumes/{catalog_name}/{schema_name}/databricksmlopspipeline/water_potability.csv"

print(f"🔍 Loading data from: {DATA_PATH}")

# Load data using Spark
water_df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

# Basic data exploration
print(f"📈 Dataset shape: {water_df.count()} rows, {len(water_df.columns)} columns")
print(f"🔍 Columns: {water_df.columns}")

# Show sample data
print("📋 Sample Data:")
water_df.show(5)

In [ ]:
# 🔧 Data Quality Validation and Preprocessing
print("🔍 DATA QUALITY ANALYSIS:")
print("-" * 40)

# Check for missing values
print("Missing values per column:")
for col_name in water_df.columns:
    null_count = water_df.filter(col(col_name).isNull() | (col(col_name) == "")).count()
    total_count = water_df.count()
    null_percentage = (null_count / total_count) * 100
    print(f"   {col_name}: {null_count} ({null_percentage:.1f}%)")

# Fill missing values with median (same as training pipeline)
print("\n🔧 Filling missing values with median...")

# Get numeric columns (exclude target)
numeric_columns = [f.name for f in water_df.schema.fields if isinstance(f.dataType, (IntegerType, FloatType, DoubleType)) and f.name != 'Potability']

# Calculate medians and fill nulls
for col_name in numeric_columns:
    median_val = water_df.select(col_name).na.drop().approxQuantile(col_name, [0.5], 0.01)[0]
    water_df = water_df.fillna({col_name: median_val})
    print(f"   ✅ {col_name}: filled with {median_val:.2f}")

print(f"\n✅ Data preprocessing complete!")

In [ ]:
# 🏗️ Add Unique Identifier and Metadata
print("🏗️ Adding metadata columns...")

# Add unique ID column (like primary key in Iris)
water_df = water_df.withColumn("sample_id", monotonically_increasing_id())

# Add ingestion timestamp
water_df = water_df.withColumn("ingestion_timestamp", current_timestamp())

# Add data quality flag
water_df = water_df.withColumn("data_quality_score", 
    col("sample_id") * 0 + 1.0  # Simple score - can be enhanced
)

print("✅ Metadata columns added: sample_id, ingestion_timestamp, data_quality_score")

# Show final schema
print("\n📋 Final Schema:")
water_df.printSchema()

In [ ]:
# 💾 Create Unity Catalog Table with Change Data Feed
TABLE_NAME = f"{catalog_name}.{schema_name}.water_quality_data"

print(f"💾 Creating Unity Catalog table: {TABLE_NAME}")

# Write to Unity Catalog with proper settings
water_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(TABLE_NAME)

print(f"✅ Table created successfully!")

# Enable Change Data Feed for monitoring (required for Lakehouse Monitor)
spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"✅ Change Data Feed enabled for monitoring")

# Create primary key constraint (like Iris)
try:
    spark.sql(f"ALTER TABLE {TABLE_NAME} ADD CONSTRAINT water_quality_pk PRIMARY KEY(sample_id)")
    print(f"✅ Primary key constraint added on sample_id")
except Exception as e:
    print(f"ℹ️ Primary key constraint already exists or not supported: {e}")

In [ ]:
# 📊 Data Validation Summary
print("🎯 DATA INGESTION SUMMARY:")
print("=" * 50)

# Count records
record_count = spark.sql(f"SELECT COUNT(*) FROM {TABLE_NAME}").collect()[0][0]
print(f"📈 Total records ingested: {record_count:,}")

# Show sample from table
print(f"\n📋 Sample records from {TABLE_NAME}:")
spark.sql(f"SELECT * FROM {TABLE_NAME} LIMIT 5").show()

# Validate data quality
potability_dist = spark.sql(f"""
    SELECT 
        Potability,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as percentage
    FROM {TABLE_NAME} 
    GROUP BY Potability 
    ORDER BY Potability
""")
print(f"\n📊 Potability Distribution:")
potability_dist.show()

print(f"\n✅ Data ingestion completed successfully!")
print(f"🎯 Next: Run model training to create Challenger model")